# Barotropic gyre: sensitivity of a box-mean temperature to the temperature field over 6 months

The tutorial barotropic gyre (62 x 62 cells of 20 km, one 5000 m layer, wind
stress -cos(y)) with temperature stepped as a passive tracer (zero thermal
expansion). The cost is the mean surface temperature over a 10 x 10-cell box over
the northern western boundary current at the end of a 180-day adjoint run that starts from
the end of a 2-year spin-up. The adjoint model writes `ADJtheta` every day: the
sensitivity of that box mean to the temperature field at each earlier day, and
`adxx_theta`, the same sensitivity at the start of the 180 days.

Figures and the animation go into the run directory on scratch (`figures/`,
`animations/`), per the repository's convention.

In [1]:
import os, re, glob
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import SymLogNorm, TwoSlopeNorm
from matplotlib.patches import Rectangle
from PIL import Image

run_dir = '/scratch2/tshahriar/barotropic_gyre_outputs/runs/adjoint/barotropic_gyre_tapAdj_ckpAll_180d_run31118'
fig_dir = os.path.join(run_dir, 'figures'); ani_dir = os.path.join(run_dir, 'animations')
os.makedirs(fig_dir, exist_ok=True); os.makedirs(ani_dir, exist_ok=True)

deltaT = 1200.0; steps_per_day = 72
box = dict(i0=5, i1=14, j0=40, j1=49)           # the cost box, 1-based global cell indices (code_tap/cost_test.F)

## Reading the MDS output
One file per iteration; the adjoint dumps are float32, the control gradient float64.

In [2]:
def read_meta(path):
    t = open(path).read()
    dims = [int(x) for x in re.search(r'dimList\s*=\s*\[([^\]]*)\]', t, re.S).group(1).replace(',', ' ').split()]
    nd = len(dims)//3
    shape = [dims[3*i+2]-dims[3*i+1]+1 for i in range(nd)][::-1]
    prec = re.search(r"dataprec\s*=\s*\[\s*'(\w+)'", t).group(1)
    nrec = int(re.search(r'nrecords\s*=\s*\[\s*(\d+)', t).group(1))
    return shape, prec, nrec

def mds_file(stem):
    """The .data file for a stem such as 'ADJtheta.0000051840' or 'XC', tiled (.001.001) or global."""
    cands = sorted(glob.glob(f'{run_dir}/{stem}.data')) + sorted(glob.glob(f'{run_dir}/{stem}.001.001.data'))
    if not cands: raise FileNotFoundError(stem)
    return cands[0]

def read_field(prefix, it):
    f = mds_file(f'{prefix}.{it:010d}')
    shape, prec, nrec = read_meta(f[:-5] + '.meta')
    a = np.fromfile(f, dtype='>f4' if prec == 'float32' else '>f8').reshape([nrec] + shape)
    return np.squeeze(a.astype('f8'))

def iterations(prefix):
    its = set()
    for f in glob.glob(f'{run_dir}/{prefix}.*.data'):
        parts = os.path.basename(f).split('.')
        if parts[1].isdigit() and len(parts[1]) == 10: its.add(int(parts[1]))
    return sorted(its)

def read_grid_var(name):
    f = mds_file(name)
    shape, prec, nrec = read_meta(f[:-5] + '.meta')
    return np.squeeze(np.fromfile(f, dtype='>f4' if prec == 'float32' else '>f8').reshape([nrec] + shape).astype('f8'))

XC = read_grid_var('XC') / 1e3; YC = read_grid_var('YC') / 1e3      # km
maskC = read_grid_var('hFacC') > 0
iters = iterations('ADJtheta'); nIter0 = iters[0]; nIterEnd = iters[-1]
days = [(it - nIter0) / steps_per_day for it in iters]
print(f'{len(iters)} ADJtheta dumps, iterations {nIter0}..{nIterEnd}, days {days[0]:.0f}..{days[-1]:.0f} of the 180')

ADJ = np.stack([read_field('ADJtheta', it) for it in iters])          # (time, y, x)
ADJ[:, ~maskC] = np.nan
adxx = read_field('adxx_theta', 0); adxx[~maskC] = np.nan

fc = None
for line in open(f'{run_dir}/output_tap_adj.txt'):
    if 'global fc' in line:
        fc = float(line.split('=')[-1]); break
print(f'cost function fc = {fc} (the box-mean temperature at day 180, in degrees C)')

180 ADJtheta dumps, iterations 51840..64728, days 0..179 of the 180
cost function fc = 21.6270244021154 (the box-mean temperature at day 180, in degrees C)


## A conservation check on the dumps
A uniform temperature change everywhere raises the box mean by the same amount,
and the passive tracer is conserved by the no-flux walls, so the sum of the
sensitivity over the ocean cells must be 1 at every lead time. The same holds
for `adxx_theta`.

In [3]:
sums = np.nansum(ADJ, axis=(1, 2))
print(f'sum of ADJtheta over the domain: min {sums.min():.6f}, max {sums.max():.6f} over {len(sums)} dumps (expected 1)')
print(f'sum of adxx_theta: {np.nansum(adxx):.6f}; adxx_theta equals ADJtheta at day 0 after float32 rounding: '
      f'{np.allclose(np.nan_to_num(adxx.astype(">f4").astype("f8")), np.nan_to_num(ADJ[0]), rtol=0, atol=0)}')

sum of ADJtheta over the domain: min 1.000000, max 1.000000 over 180 dumps (expected 1)
sum of adxx_theta: 1.000000; adxx_theta equals ADJtheta at day 0 after float32 rounding: True


## The gyre and the box

In [4]:
it_end = nIterEnd
T_end = read_field('T', it_end); Eta_end = read_field('Eta', it_end)
U_end = read_field('U', it_end); V_end = read_field('V', it_end)
T_end[~maskC] = np.nan; Eta_end[~maskC] = np.nan

def draw_box(ax):
    ax.add_patch(Rectangle((XC[0, box['i0']-1] - 10, YC[box['j0']-1, 0] - 10), 10*20, 10*20,
                           fill=False, edgecolor='k', linewidth=1.4))

fig, axes = plt.subplots(1, 2, figsize=(11, 5), constrained_layout=True)
ax = axes[0]
pc = ax.pcolormesh(XC, YC, Eta_end, cmap='RdBu_r', shading='auto')
sk = 3
ax.quiver(XC[::sk, ::sk], YC[::sk, ::sk], U_end[::sk, ::sk], V_end[::sk, ::sk], scale=0.6, width=0.003, color='k')
fig.colorbar(pc, ax=ax, label='free-surface height (m)'); draw_box(ax)
ax.set_title(f'sea surface height and velocity at day 180'); ax.set_xlabel('x (km)'); ax.set_ylabel('y (km)'); ax.set_aspect('equal')
ax = axes[1]
pc = ax.pcolormesh(XC, YC, T_end, cmap='viridis', shading='auto')
fig.colorbar(pc, ax=ax, label='temperature (degrees C)'); draw_box(ax)
ax.set_title('temperature at day 180; the cost box'); ax.set_xlabel('x (km)'); ax.set_aspect('equal')
fig.savefig(f'{fig_dir}/gyre_state_day180.png', dpi=150); plt.close(fig)

## The sensitivity at the start of the 180 days, and at selected lead times
Sensitivity of the day-180 box mean (degrees per degree) to the temperature at
the given day. A symmetric logarithmic colour scale, because the field spans
orders of magnitude.

In [5]:
vmax = np.nanmax(np.abs(ADJ)); linthresh = vmax * 1e-3
norm = SymLogNorm(linthresh=linthresh, vmin=-vmax, vmax=vmax, base=10)

fig, ax = plt.subplots(figsize=(6.2, 5.4), constrained_layout=True)
pc = ax.pcolormesh(XC, YC, adxx, cmap='RdBu_r', norm=norm, shading='auto')
fig.colorbar(pc, ax=ax, label='d(box-mean T at day 180) / dT at day 0'); draw_box(ax)
ax.set_title('adxx_theta: sensitivity to the temperature at day 0'); ax.set_xlabel('x (km)'); ax.set_ylabel('y (km)'); ax.set_aspect('equal')
fig.savefig(f'{fig_dir}/sensitivity_adxx_theta.png', dpi=150); plt.close(fig)

leads = [0, 10, 30, 60, 120, 180]
fig, axes = plt.subplots(2, 3, figsize=(14, 9), constrained_layout=True)
for ax, lead in zip(axes.ravel(), leads):
    k = int(round((180 - lead) * steps_per_day / (iters[1] - iters[0]))) if len(iters) > 1 else 0
    k = min(max(k, 0), len(iters) - 1)
    pc = ax.pcolormesh(XC, YC, ADJ[k], cmap='RdBu_r', norm=norm, shading='auto'); draw_box(ax)
    ax.set_title(f'day {days[k]:.0f}: {180 - days[k]:.0f} days before the cost'); ax.set_aspect('equal')
fig.colorbar(pc, ax=axes, label='d(box-mean T at day 180) / dT', shrink=0.7)
fig.suptitle('Sensitivity of the box-mean temperature at day 180 to the temperature field at earlier days')
fig.savefig(f'{fig_dir}/sensitivity_snapshots.png', dpi=130); plt.close(fig)

fig, ax = plt.subplots(figsize=(7, 3.2), constrained_layout=True)
ax.plot(days, np.nanmax(np.abs(ADJ), axis=(1, 2)), label='max |ADJtheta|')
ax.plot(days, [int((np.abs(ADJ[k]) > 0.01 * vmax).sum()) for k in range(len(iters))], label='cells above 1 % of the overall maximum')
ax.set_xlabel('day (the cost is measured at day 180)'); ax.set_yscale('log'); ax.legend(); ax.grid(alpha=0.3)
fig.savefig(f'{fig_dir}/sensitivity_amplitude_vs_time.png', dpi=150); plt.close(fig)

## The animation: the sensitivity from the cost time backwards
Frames run from day 180 (the cost) back to day 0, one per dump, so the plume
spreads upstream as the lead time grows.

In [6]:
frames = []
for k in range(len(iters) - 1, -1, -1):
    fig, ax = plt.subplots(figsize=(6.2, 5.6), constrained_layout=True)
    pc = ax.pcolormesh(XC, YC, ADJ[k], cmap='RdBu_r', norm=norm, shading='auto'); draw_box(ax)
    fig.colorbar(pc, ax=ax, label='d(box-mean T at day 180) / dT')
    ax.set_title(f'sensitivity to T at day {days[k]:.0f}  ({180 - days[k]:.0f} days before the cost)')
    ax.set_xlabel('x (km)'); ax.set_ylabel('y (km)'); ax.set_aspect('equal')
    fig.canvas.draw()
    frames.append(Image.fromarray(np.asarray(fig.canvas.buffer_rgba())[:, :, :3]))
    plt.close(fig)
gif = f'{ani_dir}/sensitivity_ADJtheta_6months.gif'
frames[0].save(gif, save_all=True, append_images=frames[1:], duration=80, loop=0, optimize=True)
print('wrote', gif, f'{os.path.getsize(gif)/1e6:.1f} MB, {len(frames)} frames')

wrote /scratch2/tshahriar/barotropic_gyre_outputs/runs/adjoint/barotropic_gyre_tapAdj_ckpAll_180d_run31118/animations/sensitivity_ADJtheta_6months.gif 2.9 MB, 180 frames


## The forward temperature over the same 6 months, for reference

In [7]:
T_iters = [it for it in iterations('T') if nIter0 <= it <= nIterEnd]
frames = []
for it in T_iters:
    T = read_field('T', it); T[~maskC] = np.nan
    fig, ax = plt.subplots(figsize=(6.2, 5.6), constrained_layout=True)
    pc = ax.pcolormesh(XC, YC, T, cmap='viridis', vmin=15, vmax=25, shading='auto'); draw_box(ax)
    fig.colorbar(pc, ax=ax, label='temperature (degrees C)')
    ax.set_title(f'temperature, day {(it - nIter0)/steps_per_day:.0f} of 180'); ax.set_xlabel('x (km)'); ax.set_ylabel('y (km)'); ax.set_aspect('equal')
    fig.canvas.draw(); frames.append(Image.fromarray(np.asarray(fig.canvas.buffer_rgba())[:, :, :3])); plt.close(fig)
gif2 = f'{ani_dir}/forward_temperature_6months.gif'
frames[0].save(gif2, save_all=True, append_images=frames[1:], duration=80, loop=0, optimize=True)
print('wrote', gif2, f'{os.path.getsize(gif2)/1e6:.1f} MB, {len(frames)} frames')
print('figures:', sorted(os.listdir(fig_dir)))

wrote /scratch2/tshahriar/barotropic_gyre_outputs/runs/adjoint/barotropic_gyre_tapAdj_ckpAll_180d_run31118/animations/forward_temperature_6months.gif 2.1 MB, 180 frames
figures: ['gyre_state_day180.png', 'sensitivity_adxx_theta.png', 'sensitivity_amplitude_vs_time.png', 'sensitivity_snapshots.png']
